# News Category Classifier — Web Scraping to QLoRA Fine-Tuning

Pipeline: scrape NPR articles across 5 categories → clean & explore the data →
fine-tune `meta-llama/Llama-3.2-1B` for text classification using **QLoRA**
(4-bit quantization + LoRA adapters) → evaluate → push the adapter to the
Hugging Face Hub → run inference.

**Why QLoRA instead of unfreezing raw layers:** a 1B-parameter model has
~1B trainable weights if fully unfrozen. QLoRA loads the frozen base model
in 4-bit precision and trains small low-rank adapter matrices injected into
the attention/MLP projections — typically **under 1% of the parameters**,
with accuracy close to full fine-tuning. It also means the artifact you
push to the Hub is a few megabytes, not multiple gigabytes.

### Steps
1. Crawl data
2. Parameters & config
3. Exploratory data analysis (EDA)
4. Clean data
5. Wrangle data (label encoding, stratified split, class weights, tokenization)
6. Initialize model (QLoRA)
7. Train model
8. Evaluate model
9. Model inference


# Install requirements

In [ ]:
!pip install -q datasets evaluate peft accelerate bitsandbytes seaborn

# Setup: reproducibility & logging

Fixing every seed makes the split, training run, and reported metrics
reproducible run-to-run — important both for debugging and for anyone
re-running this notebook to verify the reported numbers. Swapping `print()`
for `logging` also means log lines carry a timestamp/level and can be
filtered or redirected without touching the code that emits them.

In [ ]:
import random
import logging

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("news_pipeline")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {DEVICE}")

# Crawl Data

In [ ]:
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
import os
from getpass import getpass

In [ ]:
# Credentials are loaded once, never hardcoded, and reused by crawl()
SCRAPER_API_AUTH = os.environ.get("SCRAPER_API_AUTH") or getpass("Enter Scraper API Authorization header value: ")

def crawl(url_to_crawl: str) -> requests.Response:
    """POST a URL to the scraper API and return the raw response."""
    url = "https://scraper-api.decodo.com/v2/scrape"

    payload = {
        "url": url_to_crawl
    }

    headers = {
        "accept": "application/json",
        "content-type": "application/json",
        "authorization": SCRAPER_API_AUTH
    }

    response = requests.post(url, json=payload, headers=headers)

    return response

In [ ]:
def get_article_text(article_url: str) -> str | None:
    """Crawl a single NPR article and return its body text, or None on failure."""
    try:
        crawled_article = crawl(article_url)

        if crawled_article.status_code != 200:
            logger.debug(f"Article crawl HTTP {crawled_article.status_code} for {article_url}")
            return None

        crawled_article_json = crawled_article.json()

        if 'results' not in crawled_article_json:
            logger.debug(f"No 'results' in article response for {article_url}: {crawled_article_json}")
            return None

        status_code = crawled_article_json['results'][0]['status_code']
        if status_code != 200:
            return None

        html_string = crawled_article_json['results'][0]['content']
        soup = BeautifulSoup(html_string, 'html.parser')

        story_div = soup.find('div', id='storytext')
        if story_div is None:
            return None

        text = story_div.get_text(strip=True, separator='\n')

        return text
    except Exception as e:
        logger.debug(f"Failed to crawl article {article_url}: {e}")
        return None


In [ ]:
def get_next_article(category_url: str, batch_size: int = 10):
    """Generator that yields article text from a category's paginated index pages."""
    start_index = 1
    while True:
        crawled_page = crawl(f"{category_url}?start={start_index}&count={batch_size}")

        if crawled_page.status_code != 200:
            logger.error(
                f"Scraper API call failed (HTTP {crawled_page.status_code}) for "
                f"{category_url} at start={start_index}. Response: {crawled_page.text[:500]}"
            )
            break

        crawled_page_json = crawled_page.json()

        if 'results' not in crawled_page_json:
            logger.error(
                f"Unexpected API response shape (no 'results' key) for "
                f"{category_url} at start={start_index}. Response: {crawled_page_json}"
            )
            break

        status_code = crawled_page_json['results'][0]['status_code']
        if status_code != 200:
            break

        html_string = crawled_page_json['results'][0]['content']
        soup = BeautifulSoup(html_string, 'html.parser')

        for article in soup.find_all('article'):
            anchor_tag = article.find('a')
            if anchor_tag is None:
                continue

            article_url = anchor_tag['href']
            article_text = get_article_text(article_url)

            if article_text is None:
                continue

            yield article_text

        start_index += batch_size


In [ ]:
urls_to_crawl = {
    "politics": "https://www.npr.org/get/1014/render/partial/next",  # ?start=11&count=20
    "business": "https://www.npr.org/get/1006/render/partial/next",
    "health":   "https://www.npr.org/get/1128/render/partial/next",
    "science":  "https://www.npr.org/get/1007/render/partial/next",
    "climate":  "https://www.npr.org/get/1167/render/partial/next",
}

In [ ]:
ARTICLES_PER_CATEGORY = 1000

data = []
for news_category, category_url in urls_to_crawl.items():
    logger.info(f"Crawling {news_category}")
    articles_crawled = 0

    for article_text in get_next_article(category_url):
        data.append({'news_category': news_category, 'article': article_text})
        articles_crawled += 1

        if articles_crawled % 100 == 0:
            logger.info(f"  crawled {articles_crawled} articles")

        if articles_crawled >= ARTICLES_PER_CATEGORY:
            break

    df = pd.DataFrame(data)
    df.to_csv(f"news_articles_dataset_{news_category}.csv", index=False)

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df.to_csv("news_articles_dataset.csv", index=False)

### Steps to Fine-tune an LLM
1. Define parameters
2. Clean dataset
3. Wrangle dataset
   - Label encoder
   - Train/test split
   - Convert into Hugging Face Dataset
   - Tokenizer
4. Initialize model (QLoRA)
5. Train model
6. Evaluate model
7. Model inference

# Parameters and Reading Data

All tunable knobs live in one place — dataset columns, data-quality
thresholds, and the LoRA hyperparameters — so re-running this notebook
with a different model or a stricter quality bar means editing one cell,
not hunting through the whole file.

In [ ]:
import pandas as pd
import huggingface_hub

In [ ]:
# --- Dataset parameters ---
dataset_csv_path = 'news_articles_dataset.csv'
text_column_name = 'article'
label_column_name = 'news_category'
test_size = 0.2
MIN_ARTICLE_WORDS = 30   # drop very short scraped fragments (nav text, error pages, etc.)
num_labels = 2           # placeholder — overwritten below once we read the data

# --- Model parameters ---
model_name = 'meta-llama/Llama-3.2-1B'
hf_token = os.environ.get("HF_TOKEN") or getpass("Enter your Hugging Face token: ")

# --- QLoRA parameters ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

In [ ]:
df = pd.read_csv(dataset_csv_path)
num_labels = df[label_column_name].nunique()
logger.info(f"Loaded {len(df)} articles across {num_labels} categories")

In [ ]:
huggingface_hub.login(hf_token)

# Exploratory Data Analysis

Before cleaning or modeling, check class balance and article length. Both
feed decisions made later: an imbalanced label distribution motivates a
**stratified split** and a **class-weighted loss**, and the length
distribution helps sanity-check the `MIN_ARTICLE_WORDS` cutoff used during
cleaning.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

logger.info(f"Total articles: {len(df)}")
logger.info(f"Exact-duplicate articles: {df[text_column_name].duplicated().sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = df[label_column_name].value_counts()
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0])
axes[0].set_title("Articles per Category")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=30)

word_counts = df[text_column_name].astype(str).apply(lambda t: len(t.split()))
sns.histplot(word_counts, bins=40, ax=axes[1])
axes[1].axvline(MIN_ARTICLE_WORDS, color='red', linestyle='--', label=f'{MIN_ARTICLE_WORDS}-word cutoff')
axes[1].set_title("Article Length Distribution")
axes[1].set_xlabel("Word count")
axes[1].legend()

plt.tight_layout()
plt.show()

print(class_counts)
imbalance_ratio = class_counts.max() / class_counts.min()
logger.info(f"Class imbalance ratio (max/min): {imbalance_ratio:.2f}")

# Clean Data

In addition to stripping any leftover HTML and collapsing whitespace, drop
exact-duplicate articles (the scraper can occasionally revisit the same
story from a paginated index) and articles under `MIN_ARTICLE_WORDS` —
those are almost always scraping artifacts (nav menus, error pages) rather
than real article bodies, and they add noise without adding signal.

In [ ]:
from bs4 import BeautifulSoup
import re


class Cleaner:
    """Strips HTML remnants and normalizes whitespace in scraped article text."""

    def remove_html_tags(self, text: str) -> str:
        return BeautifulSoup(text, 'lxml').text

    def remove_double_spaces(self, text: str) -> str:
        return re.sub(' +', ' ', text)

    def clean(self, text: str) -> str:
        clean_text = self.remove_html_tags(text)
        clean_text = self.remove_double_spaces(clean_text)
        return clean_text

In [ ]:
cleaner = Cleaner()
df['text_cleaned'] = df[text_column_name].apply(cleaner.clean)

before = len(df)
df = df.drop_duplicates(subset='text_cleaned')
df = df[df['text_cleaned'].str.split().str.len() >= MIN_ARTICLE_WORDS].reset_index(drop=True)
logger.info(f"Dropped {before - len(df)} rows (duplicates / under {MIN_ARTICLE_WORDS} words) — {len(df)} remain")

# Wrangle Data

## Label Encoder

In [ ]:
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
le.fit(df[label_column_name].tolist())
df['label'] = le.transform(df[label_column_name].tolist())

## Train/Test Split

`stratify=` keeps the category proportions from the EDA step above equal in
both the train and test splits — with a plain random split, a rarer
category (e.g. `climate`) could easily end up under-represented in test,
making its reported F1 unreliable. `random_state=SEED` makes the split
reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=test_size,
    random_state=SEED,
    stratify=df[label_column_name],
)

In [ ]:
df_train.shape, df_test.shape

In [ ]:
df_train = df_train[['text_cleaned', 'label']]
df_test = df_test[['text_cleaned', 'label']]

## Class Weights

The imbalance ratio measured during EDA is passed into the training loss
later (see the `WeightedLossTrainer` in the **Train Model** section) so the
model isn't implicitly biased toward whichever category has the most
articles.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_train['label']),
    y=df_train['label'],
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
logger.info(f"Class weights: {dict(zip(le.classes_, class_weights.round(3)))}")

## Convert to Hugging Face Dataset

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
test_dataset = Dataset.from_pandas(df_test, preserve_index=False)

## Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
def preprocess_function(examples: dict) -> dict:
    return tokenizer(examples['text_cleaned'], truncation=True)

In [ ]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

# Initialize Model — QLoRA

The base model loads frozen, in 4-bit precision (`bitsandbytes`, NF4). LoRA
adapters — small trainable low-rank matrices — are then injected into the
attention and MLP projection layers (`target_modules`) using PEFT.
`modules_to_save=["score"]` keeps the classification head fully trainable:
it's randomly initialized for this task (the base checkpoint was never
trained to output 5 news categories), so it can't be left frozen like the
rest of the base model. `print_trainable_parameters()` reports exactly how
small the trainable slice is relative to the full 1B-parameter model — a
number worth quoting directly if asked "how did you make this efficient?"

4-bit quantization requires a CUDA GPU; on CPU this cell falls back to a
non-quantized LoRA setup so the notebook still runs end-to-end, just
slower.

In [ ]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

USE_4BIT = DEVICE == "cuda"

quant_config = None
if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
else:
    logger.warning("No CUDA GPU detected — training LoRA adapters without 4-bit quantization.")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    quantization_config=quant_config,
    device_map="auto" if USE_4BIT else None,
)
model.config.pad_token_id = model.config.eos_token_id

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    modules_to_save=["score"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Train Model

- **`WeightedLossTrainer`** — a small `Trainer` subclass that applies the
  class weights computed earlier to the cross-entropy loss.
- **`gradient_accumulation_steps=8`** — recovers an effective batch size of
  16 despite `per_device_train_batch_size=2` (kept small for memory
  headroom on a single GPU).
- **`load_best_model_at_end` + `EarlyStoppingCallback`** — with
  `num_train_epochs=10`, this stops training at the best-performing
  checkpoint instead of whichever epoch happens to run last, and guards
  against overfitting.
- **`metric_for_best_model="f1_macro"`** — macro-F1 weights every class
  equally, so "best" isn't dominated by whichever category has the most
  articles the way raw accuracy would be.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
import torch.nn.functional as F
import evaluate

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        **accuracy_metric.compute(predictions=predictions, references=labels),
        "f1_macro": f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"],
        "f1_weighted": f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }

In [ ]:
class WeightedLossTrainer(Trainer):
    """Trainer that applies class weights to cross-entropy loss, so the
    model isn't biased toward the majority news category."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = F.cross_entropy(logits, labels, weight=class_weights_tensor.to(logits.device))
        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=10,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    report_to="none",
    fp16=USE_4BIT,

    learning_rate=2e-4,
    weight_decay=0.01,
    seed=SEED,
    logging_steps=20,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
train_result = trainer.train()

In [ ]:
model.config.id2label = {i: label for i, label in enumerate(le.classes_)}
model.config.label2id = {label: i for i, label in enumerate(le.classes_)}

In [ ]:
# save_model() on a PEFT-wrapped model saves only the LoRA adapter weights
# (a few MB) rather than the full 1B-parameter base model
trainer.save_model('./news_classifier_model')
tokenizer.save_pretrained('./news_classifier_model')

In [ ]:
# Push in the same spirit: this uploads the adapter, not the base model —
# make sure your HF token has write access
model.push_to_hub("news-classifier-model")
tokenizer.push_to_hub("news-classifier-model")

# Evaluate Model

Beyond the per-split classification reports: a confusion matrix shows
*which* categories get confused for each other (e.g. `science` vs
`climate` is a natural overlap to expect), and the loss/F1 curves pulled
from the `Trainer`'s own log history make it possible to see whether early
stopping kicked in for a good reason — all without wiring up a separate
experiment-tracking tool.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_split(dataset, true_labels, split_name: str):
    raw_preds = trainer.predict(dataset)
    preds = np.argmax(raw_preds.predictions, axis=-1)
    print(f"--- {split_name} ---")
    print(classification_report(true_labels, preds, target_names=le.classes_))
    return preds

train_preds = evaluate_split(tokenized_train, df_train['label'].tolist(), "Train")
test_preds = evaluate_split(tokenized_test, df_test['label'].tolist(), "Test")

In [ ]:
cm = confusion_matrix(df_test['label'].tolist(), test_preds)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

In [ ]:
log_history = pd.DataFrame(trainer.state.log_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if 'loss' in log_history:
    log_history.dropna(subset=['loss']).plot(x='epoch', y='loss', ax=axes[0], legend=False, marker='o')
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")

if 'eval_f1_macro' in log_history:
    log_history.dropna(subset=['eval_f1_macro']).plot(
        x='epoch', y='eval_f1_macro', ax=axes[1], legend=False, marker='o', color='green'
    )
    axes[1].set_title("Eval Macro-F1")
    axes[1].set_xlabel("Epoch")

plt.tight_layout()
plt.show()

# Model Inference

In [ ]:
from transformers import pipeline
from huggingface_hub import whoami

hf_username = whoami()['name']
model_repo = f"{hf_username}/news-classifier-model"

clf = pipeline(
    "text-classification",
    model=model_repo,
    tokenizer=model_repo,
    top_k=None,  # return a confidence score for every category, not just the top one
)

In [ ]:
example_article = """
Naughty or nice? That's often how I think about foods packed with carbohydrates. Whole grains, like brown rice and whole wheat, fall squarely into the nice category, while white pasta and rice, well they're more naughty.

"They're naughty, in a sense, because we digest them rapidly and that creates a fast rise in blood sugar," says nutritionist Mindy Patterson, at Texas Woman's University in Houston. They're also low in fiber and protein, compared to their whole grain cousins.

Over time, all those quick surges in blood sugar can hurt your health, Patterson says. They can contribute to insulin resistance and just leave you feeling tired.
"""

In [ ]:
result = clf(example_article)
for pred in sorted(result[0], key=lambda x: -x['score']):
    print(f"{pred['label']:>12}: {pred['score']:.1%}")

# Summary

| | |
|---|---|
| **Base model** | `meta-llama/Llama-3.2-1B` |
| **Task** | 5-way news category classification (politics, business, health, science, climate) |
| **Fine-tuning technique** | QLoRA — 4-bit NF4 quantization + LoRA adapters (r=16, α=32) |
| **Trainable params** | reported by `print_trainable_parameters()` above — typically well under 1% of the base model |
| **Data handling** | deduped + length-filtered scraped articles, stratified train/test split, class-weighted loss for imbalance |
| **Model selection** | best checkpoint by macro-F1, with early stopping |
| **Artifact shipped** | LoRA adapter only (few MB) pushed to the Hugging Face Hub, separate from the frozen base model |

**Talking points this pipeline demonstrates:** parameter-efficient
fine-tuning (QLoRA) on an LLM, handling class imbalance in both the split
and the loss function, reproducible experiments (seeded, stratified),
and evaluation beyond accuracy (macro-F1, confusion matrix, training
curves) for a multi-class problem.